In [12]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
path = os.path.join( "..", "data", "processed", "accidents_geo_enriched.csv")
df = pd.read_csv(path)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580069 entries, 0 to 580068
Data columns (total 25 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   obs              580069 non-null  float64
 1   obsm             580069 non-null  float64
 2   grav             580069 non-null  int64  
 3   dep              580069 non-null  object 
 4   com              580069 non-null  object 
 5   agg              580069 non-null  int64  
 6   int              580069 non-null  float64
 7   atm              580069 non-null  float64
 8   lat              580069 non-null  float64
 9   long             580069 non-null  float64
 10  catr             580069 non-null  float64
 11  circ             580069 non-null  float64
 12  prof             580069 non-null  float64
 13  plan             580069 non-null  float64
 14  surf             580069 non-null  float64
 15  infra            580069 non-null  float64
 16  situ             580069 non-null  floa

In [13]:
df = df.drop('lum', axis=1, errors='ignore')

In [14]:
# ---------------------------------
# 1. Maillage ~10 m (DEC = 4)
# ---------------------------------
DEC = 4                                 # ≈ 11 m lat × 7 m lon @46 °
df["lat10"] = (df["lat"]  * 10**DEC).round().astype(int)
df["lon10"] = (df["long"] * 10**DEC).round().astype(int)
df["cell_id"] = df["lat10"].astype(str) + "_" + df["lon10"].astype(str)

# ---------------------------------
# 2. Agrégation
# ---------------------------------
agg_dict = {
    # Variables agrégées comme avant
    "grav":  "median",
    "vma":   lambda s: s.mode().iat[0],
    "obs":   lambda s: s.mode().iat[0],
    "obsm":   lambda s: s.mode().iat[0],
    "agg":   lambda s: s.mode().iat[0],
    "int":   lambda s: s.mode().iat[0],
    "atm":   lambda s: s.mode().iat[0],
    "catr":  lambda s: s.mode().iat[0],
    "circ":  lambda s: s.mode().iat[0],
    "prof":  lambda s: s.mode().iat[0],
    "plan":  lambda s: s.mode().iat[0],
    "surf":  lambda s: s.mode().iat[0],
    "infra": lambda s: s.mode().iat[0],
    "situ":  lambda s: s.mode().iat[0],
    "dep":   lambda s: s.mode().iat[0],
    "com":   lambda s: s.mode().iat[0],
    "TMJA":  lambda s: s.mode().iat[0],     
    "lit":  lambda s: s.mode().iat[0],     
    "bridge": lambda s: s.mode().iat[0],  
    "tunnel": lambda s: s.mode().iat[0],
    "traffic_calming": lambda s: s.mode().iat[0],
    "dist_hop_m":  "mean",
    "dist_fire_m":  "mean",
    # Compteur 
    "lat":   "count",                 # nb_accidents (on renommera)
}

grid_df = (
    df.groupby("cell_id", as_index=False)
      .agg(agg_dict)
      .rename(columns={
          "lat": "nb_accidents"})     
)

In [15]:
grid_df.head()

,cell_id,grav,vma,obs,obsm,agg,int,atm,catr,circ,...,dep,com,TMJA,lit,bridge,tunnel,traffic_calming,dist_hop_m,dist_fire_m,nb_accidents
0,414909_89708,2.0,80,0.0,2.0,1,1.0,1.0,2.0,2.0,...,2A,2A163,39672.0,no,no,no,no,991.972403,2896.945736,3
1,414945_89572,3.0,80,0.0,6.0,1,1.0,1.0,2.0,2.0,...,2A,2A163,39672.0,no,no,yes,no,1320.281832,438.778055,1
2,414963_89627,2.0,80,0.0,2.0,1,1.0,1.0,2.0,2.0,...,2A,2A163,39672.0,no,no,no,no,2427.671931,1915.973045,2
3,414981_89488,2.0,80,0.0,2.0,1,1.0,1.0,2.0,2.0,...,2A,2A272,39672.0,yes,no,no,no,4361.358876,2330.589050,2
4,414981_89501,1.5,80,0.0,2.0,1,1.0,1.0,2.0,2.0,...,2A,2A163,39672.0,yes,no,no,no,455.291323,3303.309305,2


In [16]:
grid_df.to_csv("../data/processed/maillage_geo_enriched.csv", index=False)